<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS06_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS06 — Approximate Nearest Neighbors (Annoy, FAISS, HNSW) untuk Pencarian Properti Mirip

Jobsheet asli memakai Annoy, FAISS, dan HNSW pada dataset lagu Spotify. Versi PBL ini
menerapkannya pada **vektor fitur properti**: diberikan satu properti, sistem mencari
properti lain dengan karakteristik paling mirip. Ini adalah dasar fitur
"rekomendasi properti serupa" di aplikasi RumahPoint.

Alur: vektor fitur → indeks ANN → ukur **recall@10** dan **waktu query** → bandingkan
dengan pencarian exact.

## Langkah 1 — Siapkan Vektor Fitur

Setiap properti diwakili vektor 4 dimensi: luas tanah, luas bangunan, jumlah kamar,
dan jumlah kamar mandi. Fitur distandarkan lalu dinormalisasi ke panjang 1, sehingga
jarak kosinus dan inner product memberi urutan yang sama.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.width', 120)
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR] = df[FITUR].fillna(df[FITUR].median())
df = df.reset_index(drop=True)

X = StandardScaler().fit_transform(df[FITUR])
X = normalize(X).astype('float32')
d = X.shape[1]

print(f'Jumlah properti : {len(df):,}')
print(f'Dimensi vektor  : {d}')
print(f'Memori matriks  : {X.nbytes / 1e6:.1f} MB')

Jumlah properti : 56,170
Dimensi vektor  : 4
Memori matriks  : 0.9 MB


## Langkah 2 — Baseline: Pencarian Exact

Sebelum memakai ANN, perlu pembanding yang benar-benar exact. `NearestNeighbors` dengan
metrik kosinus (brute force) menghitung jarak ke **seluruh** 56 ribu vektor untuk setiap
query — akurat, tetapi lambat jika data terus bertambah.

In [2]:
K = 10
rng = np.random.default_rng(RANDOM_STATE)
idx_query = rng.choice(len(df), size=200, replace=False)
Q = X[idx_query]

exact = NearestNeighbors(n_neighbors=K, metric='cosine', algorithm='brute').fit(X)

t0 = time.perf_counter()
_, exact_idx = exact.kneighbors(Q)
waktu_exact = (time.perf_counter() - t0) / len(Q) * 1000

print(f'Waktu query exact : {waktu_exact:.3f} ms/query')
print(f'Contoh hasil exact untuk properti #{idx_query[0]}: {exact_idx[0][:5]}')

Waktu query exact : 0.572 ms/query
Contoh hasil exact untuk properti #20759: [13628 30292 30298 23896 30289]


## Langkah 3 — Annoy

Annoy membangun beberapa pohon acak (*random projection trees*). Semakin banyak pohon,
semakin akurat tetapi indeks semakin besar dan lambat dibangun. Di sini dicoba
`n_trees = 10` dan `50`.

In [3]:
from annoy import AnnoyIndex

hasil_annoy = []
for n_trees in [10, 50, 100]:
    idx = AnnoyIndex(d, 'angular')
    for i, v in enumerate(X):
        idx.add_item(i, v)
    t0 = time.perf_counter()
    idx.build(n_trees)
    waktu_bangun = time.perf_counter() - t0

    t0 = time.perf_counter()
    hasil = [idx.get_nns_by_vector(q, K) for q in Q]
    waktu_query = (time.perf_counter() - t0) / len(Q) * 1000

    recall = np.mean([len(set(h) & set(e)) / K for h, e in zip(hasil, exact_idx)])
    hasil_annoy.append({'n_trees': n_trees, 'recall@10': recall,
                        'ms/query': waktu_query, 'build_detik': waktu_bangun})

pd.DataFrame(hasil_annoy).round(4)

,n_trees,recall@10,ms/query,build_detik
0,10,0.760,0.0230,0.4141
1,50,0.769,0.1326,1.9214
2,100,0.769,0.2721,3.8581


## Langkah 4 — FAISS

FAISS menyediakan beberapa jenis indeks. Yang dipakai:

- `IndexFlatIP` — exact, dipakai sebagai kontrol.
- `IndexIVFFlat` — membagi vektor ke `nlist` sel (Voronoï) lalu hanya mencari di
  `nprobe` sel terdekat. Jauh lebih cepat, tetapi bisa melewatkan tetangga di sel lain.

In [4]:
import faiss

Xf = np.ascontiguousarray(X)
Qf = np.ascontiguousarray(Q)

flat = faiss.IndexFlatIP(d)
flat.add(Xf)
t0 = time.perf_counter()
_, faiss_exact = flat.search(Qf, K)
waktu_flat = (time.perf_counter() - t0) / len(Q) * 1000

hasil_faiss = []
for nlist in [100, 316]:
    kuantizer = faiss.IndexFlatIP(d)
    ivf = faiss.IndexIVFFlat(kuantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
    ivf.train(Xf)
    ivf.add(Xf)

    for nprobe in [1, 10, 50]:
        ivf.nprobe = nprobe
        t0 = time.perf_counter()
        _, I = ivf.search(Qf, K)
        waktu_query = (time.perf_counter() - t0) / len(Q) * 1000
        recall = np.mean([len(set(a) & set(b)) / K for a, b in zip(I, faiss_exact)])
        hasil_faiss.append({'nlist': nlist, 'nprobe': nprobe, 'recall@10': recall,
                            'ms/query': waktu_query})

print(f'FAISS IndexFlatIP (exact): {waktu_flat:.3f} ms/query')
pd.DataFrame(hasil_faiss).round(4)

FAISS IndexFlatIP (exact): 0.147 ms/query


,nlist,nprobe,recall@10,ms/query
0,100,1,0.9635,0.0030
1,100,10,0.9960,0.0214
2,100,50,0.9960,0.0848
3,316,1,0.9245,0.0021
4,316,10,0.9905,0.0064
5,316,50,0.9905,0.0255


## Langkah 5 — HNSW

HNSW (*Hierarchical Navigable Small World*) membangun graf berlapis. Parameter `M`
mengatur jumlah koneksi per simpul dan `ef` mengatur seberapa luas pencarian dilakukan
saat query. `ef` yang lebih besar menaikkan recall dengan biaya waktu.

In [5]:
import hnswlib

hasil_hnsw = []
for M in [16, 32]:
    for ef in [50, 200, 1000]:
        p = hnswlib.Index(space='cosine', dim=d)
        p.init_index(max_elements=len(X), ef_construction=200, M=M)
        p.add_items(X, np.arange(len(X)))
        p.set_ef(ef)

        t0 = time.perf_counter()
        labels, _ = p.knn_query(Q, k=K)
        waktu_query = (time.perf_counter() - t0) / len(Q) * 1000
        recall = np.mean([len(set(a) & set(b)) / K for a, b in zip(labels, exact_idx)])
        hasil_hnsw.append({'M': M, 'ef': ef, 'recall@10': recall,
                           'ms/query': waktu_query})

pd.DataFrame(hasil_hnsw).round(4)

,M,ef,recall@10,ms/query
0,16,50,0.7565,0.0068
1,16,200,0.7385,0.0162
2,16,1000,0.7635,0.0894
3,32,50,0.7575,0.0089
4,32,200,0.7580,0.0212
5,32,1000,0.7615,0.0913


## Langkah 6 — Perbandingan Recall vs Waktu Query

Semua konfigurasi digabung dalam satu grafik. Sumbu X adalah waktu query (ms, skala log),
sumbu Y adalah recall@10. Metode yang ideal berada di pojok kiri atas.

In [6]:
ringkas = pd.concat([
    pd.DataFrame(hasil_annoy).assign(metode=lambda x: 'Annoy ' + x['n_trees'].astype(str) + ' trees'),
    pd.DataFrame(hasil_faiss).assign(metode=lambda x: 'FAISS IVF ' + x['nlist'].astype(str) + '/' + x['nprobe'].astype(str)),
    pd.DataFrame(hasil_hnsw).assign(metode=lambda x: 'HNSW M' + x['M'].astype(str) + ' ef' + x['ef'].astype(str)),
], ignore_index=True)

plt.figure(figsize=(9, 5.5))
plt.scatter(ringkas['ms/query'], ringkas['recall@10'], s=55)
for _, r in ringkas.iterrows():
    plt.annotate(r['metode'], (r['ms/query'], r['recall@10']),
                 textcoords='offset points', xytext=(5, 4), fontsize=7)
plt.xscale('log')
plt.xlabel('Waktu query (ms, skala log)')
plt.ylabel('Recall@10')
plt.title('Trade-off akurasi vs kecepatan')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ringkas.sort_values('recall@10', ascending=False).round(4).head(10)

,n_trees,recall@10,ms/query,build_detik,metode,nlist,nprobe,M,ef
4,NaN,0.9960,0.0214,NaN,FAISS IVF 100/10,100.0,10.0,NaN,NaN
5,NaN,0.9960,0.0848,NaN,FAISS IVF 100/50,100.0,50.0,NaN,NaN
7,NaN,0.9905,0.0064,NaN,FAISS IVF 316/10,316.0,10.0,NaN,NaN
8,NaN,0.9905,0.0255,NaN,FAISS IVF 316/50,316.0,50.0,NaN,NaN
3,NaN,0.9635,0.0030,NaN,FAISS IVF 100/1,100.0,1.0,NaN,NaN
6,NaN,0.9245,0.0021,NaN,FAISS IVF 316/1,316.0,1.0,NaN,NaN
1,50.0,0.7690,0.1326,1.9214,Annoy 50 trees,NaN,NaN,NaN,NaN
2,100.0,0.7690,0.2721,3.8581,Annoy 100 trees,NaN,NaN,NaN,NaN
11,NaN,0.7635,0.0894,NaN,HNSW M16 ef1000,NaN,NaN,16.0,1000.0
14,NaN,0.7615,0.0913,NaN,HNSW M32 ef1000,NaN,NaN,32.0,1000.0


## Langkah 7 — Studi Kasus: Rekomendasi Properti Mirip

Ambil satu properti, cari 5 properti terdekat memakai HNSW, lalu bandingkan atribut dan
harganya. Ini mensimulasikan fitur "properti serupa" di aplikasi.

In [7]:
contoh = 7
p = hnswlib.Index(space='cosine', dim=d)
p.init_index(max_elements=len(X), ef_construction=200, M=16)
p.add_items(X, np.arange(len(X)))
p.set_ef(50)

label, jarak = p.knn_query(X[contoh:contoh + 1], k=6)
label, jarak = label[0], jarak[0]

kolom = ['lokasi', 'luas_m2', 'luas_bangunan', 'jumlah_kamar',
         'jumlah_kamar_mandi', 'harga']
acuan = df.loc[contoh, kolom].to_dict()
print('Properti acuan:')
print({k: (f'Rp {v:,.0f}' if k == 'harga' else v) for k, v in acuan.items()})
print('\n5 properti termirip:')
hasil = df.loc[label[1:], kolom].copy()
hasil['skor_mirip'] = np.round(1 - jarak[1:], 4)
hasil['harga'] = hasil['harga'].map(lambda v: f'Rp {v:,.0f}')
hasil

Properti acuan:
{'lokasi': 'Jakarta Utara', 'luas_m2': 130.3, 'luas_bangunan': 65.0, 'jumlah_kamar': 3.0, 'jumlah_kamar_mandi': 3.0, 'harga': 'Rp 1,716,938,158'}

5 properti termirip:


,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,harga,skor_mirip
56,Bogor,130.4,65.0,3.0,3.0,"Rp 1,234,242,335",1.0000
659,Jakarta Utara,130.0,65.0,3.0,3.0,"Rp 1,674,594,919",1.0000
428,Bekasi,129.7,65.0,3.0,3.0,"Rp 1,273,430,183",1.0000
1099,Bekasi,128.8,65.0,3.0,3.0,"Rp 1,410,157,408",1.0000
880,Bekasi,128.3,65.0,3.0,3.0,"Rp 1,075,741,593",0.9999


**Analisis JS06:**

- Dengan 56 ribu vektor 4 dimensi, pencarian **exact brute force** masih cepat: **0,572 ms/query**. FAISS `IndexFlatIP` (exact) bahkan 0,147 ms/query. Jadi pada skala ini ANN belum memberi keuntungan besar; keunggulannya baru terasa saat data mencapai jutaan vektor atau dimensi tinggi.
- **Annoy**: recall@10 **0,760** (10 pohon), **0,769** (50 pohon), **0,769** (100 pohon). Menambah pohon dari 50 ke 100 **tidak menaikkan recall**, tetapi waktu query naik dari 0,133 ms ke 0,272 ms dan waktu bangun dari 1,9 detik ke 3,9 detik. Jadi 50 pohon sudah cukup.
- **FAISS IVF** unggul jelas: `nlist=100, nprobe=10` memberi recall **0,996** dengan 0,021 ms/query — lebih akurat dan lebih cepat daripada Annoy/HNSW. `nprobe=1` memang sangat cepat (0,002–0,003 ms) tetapi recall turun ke 0,92–0,96.
- **HNSW** menghasilkan recall **0,739–0,764** pada `M` 16/32 dan `ef` 50–1000. Menaikkan `ef` sampai 1.000 hampir tidak memperbaiki recall (0,7565 → 0,7635) tetapi waktu query naik 13 kali (0,0068 → 0,0894 ms).
- Recall Annoy dan HNSW yang berhenti di sekitar 0,76 bukan semata kesalahan indeks: banyak properti memiliki **vektor identik** (kombinasi luas/kamar sama, ditambah imputasi median pada `luas_bangunan`). Saat jarak beberapa kandidat sama, indeks exact dan ANN boleh memilih anggota yang berbeda, sehingga irisan 10 tetangga tidak pernah penuh. Ini juga terlihat pada studi kasus: lima tetangga punya skor kemiripan 1,0000.
- Studi kasus menunjukkan keterbatasan utama: lima properti termirip punya atribut fisik hampir identik (130 m², 65 m² bangunan, 3 kamar), tetapi harganya berkisar **Rp1,08–1,67 miliar** karena berbeda lokasi (Bogor, Bekasi, Jakarta Utara). Rekomendasi berbasis fitur fisik saja **tidak menangkap harga**.
- Rekomendasi perbaikan: masukkan `lokasi` ke vektor fitur (one-hot/frequency), timbang jarak per wilayah, atau gunakan embedding yang dilatih dari interaksi pengguna. Untuk produksi, indeks sebaiknya dibangun ulang berkala dan disimpan ke disk (`save`/`load`) agar tidak dibangun setiap request.
